#### config

In [ ]:
!pip install sentence-transformers faiss-cpu pandas numpy tqdm python-dotenv >> /dev/null

In [39]:
from __future__ import annotations
from pathlib import Path
import json, os, pickle, warnings, rich
from dataclasses import dataclass
from typing import Any, Literal
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import numpy.typing as npt
from sentence_transformers import SentenceTransformer
import faiss

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore",message="BertModel LOAD REPORT*")

#  position_ids.UNEXPECTED displays on loading the faiss index
#  can be ignored, since the model used is based on bert which requires this parameter;
#    but its configured since its a pretrained model

In [40]:
@dataclass()
class Config:
    data_dir = "/home/ronji/Dev/CarP/esco_dataset/formatted/"
    db_dir = "db/"
    model_hf = "paraphrase-multilingual-MiniLM-L12-v2"
    HF_TOKEN = os.getenv('HF_TOKEN')

    # embedding configuration
    batch_size = 256  # size of an encoding batch
    normalize_embeddings = True  # enable cosine similarity (inner product)
    index_type = "flat"  # store and compare all vectors directly, used instead of IFV (index by similarity, for large datasets)


config = Config()

### Record class
A single record corresponds to one ESCO entity

In [41]:
EntityType = Literal[
    "occupation",
    "skill",
    "isco_group",  # categorisation
    "skill_group"
]


@dataclass
class Label:
    entity_type: EntityType
    id: str
    preferred_label: str
    alt_labels: list[str]
    description: str
    code: str
    isco_code: str

    @property
    def embedding_text(self) -> str:
        text = self.preferred_label
        if self.alt_labels:
            joined = " / ".join(self.alt_labels)
            text = "{} | {}".format(text, joined)
        if self.description:
            text = "{} | {}".format(text, self.description[:250]) # shorten for context
        return text

    def to_dict(self) -> dict[str, Any]:
        return {
            "entity_type": self.entity_type,
            "id": self.id,
            "preferred_label": self.preferred_label,
            "alt_labels": self.alt_labels,
            "description": self.description,
            "isco_code": self.isco_code
        }


### Data parsing

In [7]:
def get_alt_labels(col_data):
    return [] if pd.isna(col_data) \
        else [jobtitle.strip() for jobtitle in col_data.split("\n")]

"""
there is an additional "UUIDHISTORY" column in the datasets, which is generated by tabiya data transformation script I rewrote to python to use in this project.
TODO: figure out if its safe to remove
"""

def load_occupation_data(occ_path) -> list[Label]:
    # columns: ESCOURI,ID,UUIDHISTORY,PREFERREDLABEL,ALTLABELS,DESCRIPTION,ISCOGROUPCODE,CODE
    df = pd.read_csv(occ_path)
    df = df.fillna("")

    records = []
    for _,row in df.iterrows():
        records.append(Label(
            entity_type="occupation",
            esco_uri=row.get("ESCOURI"),
            id=row.get("ID"),
            preferred_label=row.get("PREFERREDLABEL"),
            alt_labels=get_alt_labels(row.get("ALTLABELS")),
            description=row.get("DESCRIPTION"),
            code=row.get("CODE"), # more specific than isco code
            isco_code=row.get("ISCOGROUPCODE")
        ))
    return records


def load_skill_data(skill_path) -> list[Label]:
    # columns: ESCOURI,ID,UUIDHISTORY,PREFERREDLABEL,ALTLABELS,DESCRIPTION
    df = pd.read_csv(skill_path)
    df = df.fillna("")

    records = []
    for _,row in df.iterrows():
        records.append(Label(
            entity_type="skill",
            esco_uri=row.get("ESCOURI"),
            id=row.get("ID"),
            preferred_label=row.get("PREFERREDLABEL"),
            alt_labels=get_alt_labels(row.get("ALTLABELS")),
            description=row.get("DESCRIPTION"),
            code="", # just so I can use single data class for labels because splitting a vector database in more sections is more complexity
            isco_code=""
        ))
    return records

def load_isco(isco_path) -> list[Label]:
    # columns: ESCOURI, ID, UUIDHISTORY, CODE,PREFERREDLABEL

    df: pd.DataFrame = pd.read_csv(isco_path)
    df = df.fillna("")

    records = []
    for _, row in df.iterrows():
        records.append(
            Label(
                entity_type="isco_group",
                esco_uri= row.get("ESCOURI"),
                id= row.get("ID"),
                preferred_label= row.get("PREFERREDLABEL"),
                alt_labels=[],
                description="",
                code=row.get("CODE"),
                isco_code=""
            )
        )
    return records

def load_skill_groups(sg_path) -> list[Label]:
    # columns: ESCOURI,ID,UUIDHISTORY,CODE,PREFERREDLABEL
    df: pd.DataFrame = pd.read_csv(sg_path, dtype=str, keep_default_na=False)
    df = df.fillna("")
    records = []
    for _, row in df.iterrows():
        records.append(
            Label(
                entity_type="skill_group",
                esco_uri= row.get("ESCOURI"),
                id= row.get("ID"),
                preferred_label= row.get("PREFERREDLABEL"),
                code=row.get("CODE"),
                alt_labels=[],
                description="",
                isco_code=""
            )
        )
    return records

In [8]:
occupations_path = config.data_dir + "combinedOccupations_cs.csv"
skills_path = config.data_dir + "combinedSkills_cs.csv"
iscogroups_path = config.data_dir + "ISCOGroups_cs.csv"
skillgroups_path = config.data_dir + "skillGroups_cs.csv"
labels: list[Label] = []

occ_labels = load_occupation_data(occupations_path)
labels.extend(occ_labels)

skill_labels = load_skill_data(skills_path)
labels.extend(skill_labels)

isco_labels = load_isco(iscogroups_path)
labels.extend(isco_labels)

skillgroup_labels = load_skill_groups(skillgroups_path)
labels.extend(skillgroup_labels)

print(f"{len(labels)} label records")

20441 label records


In [9]:
import random

print(labels[random.randint(0,len(labels))].embedding_text)

obsluha řídícího centra ropné rafinérie | operátorka velínu ropné rafinérie / operátor velínu ropné rafinérie | Obsluha řídícího centra ropné rafinérie provádí řadu úkolů z kontrolní místnosti rafinerie. Monitoruje procesy prostřednictvím elektronických zobrazení na monitorech, číselnících a ukazatelích. Obsluha řídícího centra provádí změny proměnných a komun


In [10]:
load_dotenv()

model = SentenceTransformer(model_name_or_path=config.model_hf,token=os.getenv('HF_TOKEN'))
embedding_dimension = model.get_sentence_embedding_dimension()
# TODO: notes about dimension and pooling

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Create vector db

##### \> Encode text segments into embeddings

In [11]:
texts = [l.embedding_text for l in labels]

embeddings = model.encode(
    texts,
    batch_size=config.batch_size,
    show_progress_bar=True,
    normalize_embeddings=config.normalize_embeddings,  # L2-normalise for cosine similarity
    convert_to_numpy=True
)

embeddings = embeddings.astype(np.float32) # faiss needs the numpy type.

print(f'{embeddings.shape}')

Batches:   0%|          | 0/80 [00:00<?, ?it/s]

(20441, 384)


##### \> Build a database index from a matrix of embedding vectors of esco entities

In [13]:
d = embeddings.shape[1]
index = faiss.IndexFlatIP(d)
index.add(embeddings)

print(f'{index.ntotal}')

20441


In [16]:
faiss.write_index(index,f'{str(Path(config.db_dir))}/esco_similarity.index')

##### \> map metadata (actual text) to vectors

In [17]:
metadata = [l.to_dict() for l in labels]

with open(f'{config.db_dir}/metadata.json','w',encoding='utf-8') as w:
    json.dump(metadata,w,ensure_ascii=False,indent=4)

In [19]:
id_idx = {label.id: idx for idx, label in enumerate(labels)}

with open(f'{config.db_dir}/id_to_index.pkl', "wb") as f: # actual file for data lookup by index of returned vector
    pickle.dump(id_idx, f)

### Interface

In [42]:
@dataclass
class Result:
    rank: int   # 1-based rank
    cosine_score: float # higher = more similar
    entity_type: str
    label: str
    id: str
    code: str
    isco_code: str
    description: str


def search(
    query, # extracted piece of text
    model,
    index,
    metadata: list[dict[str, Any]], # columns that are null for some labels
    top_n: int = 10, # results to display
    entity_type: Optional[EntityType] = None, # filter by entity type
) -> list[Result]:

    query_vector = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype(np.float32)

    # Fetch more candidates if theres filtering - TODO
    fetch_k: int = top_n * 5 if entity_type else top_n

    # search the faiss db (inner product = cosine similarity for normalised vectors)
    scores: npt.NDArray[np.float32]
    indices: npt.NDArray[np.int64]
    results = {}

    while len(results) < top_n:
        scores, indices = index.search(query_vector, fetch_k + len(results))
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1: continue  # FAISS returns -1 for empty slots
            data = metadata[int(idx)]

            if (entity_type and data["entity_type"] != entity_type) or data["id"] in results:
                continue

            results[data["id"]] = Result(
                rank=len(results) + 1,
                cosine_score=float(score),
                entity_type=data["entity_type"],
                label=data["preferred_label"],
                id=data["id"],
                code=data.get("code", ""),
                isco_code=data.get("isco_code",""),
                description=data.get("description",""),
                )

    return list(results.values())


def print_results(results):
    for r in results:
        rich.print(r)
        print()

In [43]:
def load_db(): # entry function to use in calls
    model: SentenceTransformer = SentenceTransformer(config.model_hf, token=os.getenv('HF_TOKEN'))
    index = faiss.read_index(f'{config.db_dir}/esco_similarity.index')
    with open(f'{config.db_dir}/metadata.json', "r", encoding="utf-8") as f:
        metadata = json.load(f)

    return model, index, metadata

### testing examples

In [ ]:
model, db,metadata = load_db()

# TODO: parse out \axa0f in metadata
# TODO: separate into databases by label for performance

##### all

In [ ]:
q = input('query all types: ')
results = search(q, model, db, metadata, top_n=5)
print_results(results)

##### occupations

In [ ]:
q_occ = input('query occupations: ')
results = search(q_occ, model, db, metadata, top_n=5, entity_type="occupation")
print_results(results)

##### skills

In [46]:
q_s = input('query skills: ')
results = search(q_s, model, db, metadata, top_n=5, entity_type="skill")
print_results(results)

Result(
    rank=1,
    cosine_score=0.6571429967880249,
    entity_type='skill',
    label='kancelářský software',
    id='key_17523',
    code='',
    isco_code='',
    description='Vlastnosti a\xa0fungování softwarových programů pro kancelářské práce, jako je zpracování textů, 
tabulky, prezentace, e-maily a\xa0databáze.'
)

Result(
    rank=2,
    cosine_score=0.6571429967880249,
    entity_type='skill',
    label='kancelářský software',
    id='key_14865',
    code='',
    isco_code='',
    description='Vlastnosti a\xa0fungování softwarových programů pro kancelářské práce, jako je zpracování textů, 
tabulky, prezentace, e-maily a\xa0databáze.'
)

Result(
    rank=3,
    cosine_score=0.57850182056427,
    entity_type='skill',
    label='zajišťovat kancelářské vybavení',
    id='key_14645',
    code='',
    isco_code='',
    description='Sledovat, analyzovat a\xa0poskytovat zařízení potřebná v\xa0kancelářích a\xa0obchodních zařízeních
k zajištění hladkého průběhu operací. Připravovat přístroje, jako jsou komunikační zařízení, počítače, faxy 
a\xa0fotokopírky.'
)

Result(
    rank=4,
    cosine_score=0.577736496925354,
    entity_type='skill',
    label='používat aplikaci Microsoft Office',
    id='key_16988',
    code='',
    isco_code='',
    description='Používat běžné programy obsažené v balíčku Microsoft Office. Vytvořit dokument a provádět základní
formátování, vkládat zalomení stránky, vytvářet záhlaví nebo zápatí a vkládat grafické prvky, vytvářet automaticky 
generovaný obsah a slučovat dopisové formuláře z databáze adres. Vytvářet tabulky s automatizovanými výpočty, 
vytvářet obrázky a třídit a filtrovat tabulky s údaji.\n   '
)

Result(
    rank=5,
    cosine_score=0.5602760314941406,
    entity_type='skill',
    label='používat kancelářské systémy',
    id='key_9884',
    code='',
    isco_code='',
    description='Vhodně a\xa0včas využívat kancelářské systémy používané v\xa0obchodních zařízeních v\xa0závislosti
na\xa0cíli, ať už se jedná o\xa0sběr zpráv, uchovávání informací o\xa0klientovi nebo plánování agendy. To zahrnuje 
správu systémů, jako je řízení vztahů se zákazníky, správa dodavatelů, ukládání dat a systémy hlasové schránky.'
)

Result(
    rank=6,
    cosine_score=0.5554985404014587,
    entity_type='skill',
    label='řídit kancelářské systémy',
    id='key_9494',
    code='',
    isco_code='',
    description='Udržovat správu a\xa0provozuschopnost jednotlivých kancelářských systémů potřebných pro hladký 
každodenní provoz kancelářských prostor, jako jsou systémy pro interní komunikaci, software pro společné používání 
uvnitř společnosti a\xa0kancelářské sítě.'
)

Result(
    rank=7,
    cosine_score=0.5334762334823608,
    entity_type='skill',
    label='výroba kancelářského vybavení',
    id='key_16385',
    code='',
    isco_code='',
    description='Výroba kalkulaček, sešívaček, tiskových kazet, vázacího vybavení, fotokopírek, tabulí a\xa0všech 
typů vybavení a\xa0přístrojů používaných v\xa0kanceláři.'
)

Result(
    rank=8,
    cosine_score=0.5279282331466675,
    entity_type='skill',
    label='kancelářské vybavení',
    id='key_8561',
    code='',
    isco_code='',
    description='Nabízené kancelářské stroje a\xa0vybavení, jejich funkce, vlastnosti a\xa0právní a\xa0regulační 
požadavky.'
)

Result(
    rank=9,
    cosine_score=0.5236691236495972,
    entity_type='skill',
    label='dohlížet na pracovníky v provozovně sázkové kanceláře',
    id='key_9239',
    code='',
    isco_code='',
    description='Sledovat, kontrolovat a\xa0plánovat každodenní úkoly zaměstnanců sázkové kanceláře.'
)